# toolroute-06b: train and evaluate a LoRA adapter for Qwen3-0.6B (GPU required)

**Run this on a GPU runtime** (Colab: Runtime → Change runtime type → T4 GPU). It stops if no GPU is present;
do not fine-tune on CPU.

Order: base-model eval (val) → smoke train (~20 steps) → full train → adapter eval (val) → **one** test eval for base
and adapter with identical generation settings → end-to-end harness check → optional publish (only after evaluation).
The test split is used once. Do not change prompts, settings or checkpoints after looking at test results.

In [ ]:
import subprocess, torch
assert torch.cuda.is_available(), "No GPU: switch the runtime to a GPU. Do not fine-tune on CPU."
print(torch.cuda.get_device_name(0), "bf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
!git clone -q https://github.com/Pranjulcr7/toolroute-06b
%cd toolroute-06b
!pip -q install -e ".[train]"
!pytest -q

In [ ]:
# Pin the exact base revision and record it.
from huggingface_hub import HfApi
BASE = "Qwen/Qwen3-0.6B"
REV = HfApi().model_info(BASE).sha
print(BASE, REV)

In [ ]:
# Persist checkpoints outside the ephemeral VM (optional but recommended).
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = "/content/drive/MyDrive/toolroute-06b/runs/lora-v1"
!mkdir -p results

## 1. Base model on validation (sanity check of prompt and parsing)

In [ ]:
!toolroute predict --base {BASE} --revision {REV} --split val --out results/base-val.json

## 2. Smoke train: ~20 optimizer steps. It checks the real tokenizer's loss masks and EOS and estimates time per step

In [ ]:
from toolroute.train import TrainConfig, run
smoke = run(TrainConfig(revision=REV, out_dir="/content/smoke", epochs=1, max_train_examples=320,
                        max_eval_examples=32, eval_steps=10))
print(smoke["loss_mask_check"]["example"])       # trained text must be the JSON target + <|im_end|>
print(smoke["seq_len"], smoke["wall_time_s"], "s", smoke["peak_memory_gb"], "GB")
est_min = smoke["wall_time_s"] / 320 * 1991 * 2 / 60
print(f"estimated full run: ~{est_min:.0f} min for 2 epochs")

## 3. Full run (checkpoint selection: lowest validation loss)

In [ ]:
rec = run(TrainConfig(revision=REV, out_dir=RUN_DIR, epochs=2))
print({k: rec[k] for k in ("device", "precision", "wall_time_s", "peak_memory_gb", "best_eval_loss", "best_checkpoint")})

## 4. Adapter on validation, then ONE test evaluation for both models (same settings)

In [ ]:
!toolroute predict --base {BASE} --revision {REV} --adapter {RUN_DIR}/adapter --split val --out results/lora-val.json
!toolroute predict --base {BASE} --revision {REV} --split test --out results/base-test.json
!toolroute predict --base {BASE} --revision {REV} --adapter {RUN_DIR}/adapter --split test --out results/lora-test.json
!toolroute baseline --split test --out results/router-test.json

In [ ]:
import json
for name in ["router-test", "base-test.score", "lora-test.score"]:
    s = json.load(open(f"results/{name}.json"))["summary"]
    print(name, {k: s[k] for k in ["exact_match", "exact_match_ci95", "parse_valid", "tool_choice_accuracy",
                                   "argument_accuracy", "unauthorized_action_rate"]},
          "clarify F1", s["clarify"]["f1"], "decline F1", s["decline"]["f1"], "heldout", s["heldout_structure"])

## 5. End-to-end in the agent-reliability-lab harness (21 held-out multi-step episodes)

In [ ]:
!toolroute e2e --model hf --base {BASE} --revision {REV} --out results/e2e-base.json
!toolroute e2e --model hf --base {BASE} --revision {REV} --adapter {RUN_DIR}/adapter --out results/e2e-lora.json

## 6. Publish (only if steps 3–5 completed), then verify loading in a fresh process

* Log in with `notebook_login()`. The token is entered in a masked prompt; never paste it into a cell.
* Fill the model card from the **measured** numbers above. If the adapter did not beat the base model, publish that
  result as it is, or do not publish the adapter at all.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo("PranjulGupta/incident-tool-routing-v1", repo_type="dataset", exist_ok=True)
api.upload_folder(folder_path="data/v1", repo_id="PranjulGupta/incident-tool-routing-v1", repo_type="dataset")
api.upload_file(path_or_fileobj="cards/DATASET_CARD.md", path_in_repo="README.md",
                repo_id="PranjulGupta/incident-tool-routing-v1", repo_type="dataset")
# Adapter: edit cards/MODEL_CARD.md with the measured results and REV first.
api.create_repo("PranjulGupta/ToolRoute-Qwen3-0.6B-LoRA", exist_ok=True)
api.upload_folder(folder_path=f"{RUN_DIR}/adapter", repo_id="PranjulGupta/ToolRoute-Qwen3-0.6B-LoRA",
                  ignore_patterns=["training_args.bin"])
api.upload_file(path_or_fileobj="cards/MODEL_CARD.md", path_in_repo="README.md",
                repo_id="PranjulGupta/ToolRoute-Qwen3-0.6B-LoRA")

In [ ]:
# Fresh-process load of the PUBLISHED adapter with the documented base revision.
script = f'''
import torch, json
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from toolroute.format import messages
tok = AutoTokenizer.from_pretrained("PranjulGupta/ToolRoute-Qwen3-0.6B-LoRA")
base = AutoModelForCausalLM.from_pretrained("{BASE}", revision="{REV}", dtype=torch.float16).cuda()
model = PeftModel.from_pretrained(base, "PranjulGupta/ToolRoute-Qwen3-0.6B-LoRA").eval()
p = tok.apply_chat_template(messages("What version is orders-api running?", None, []), tokenize=False,
                            add_generation_prompt=True, enable_thinking=False)
ids = tok(p, return_tensors="pt").to("cuda")
out = model.generate(**ids, max_new_tokens=80, do_sample=False)
print(tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))
'''
open("/content/load_check.py", "w").write(script)
!python /content/load_check.py